# EDA avanzado del dataset multimodal completo

Objetivo: auditar `creative_feature_base_tabular_visual_cv.parquet` para decidir que columnas quitar, que columnas conservar solo como metadata/outcomes, y que columnas usar como base inicial para CBR.

Regla central: una columna puede estar en el dataset final de casos, pero no necesariamente debe ser feature de similitud/modelo. Separaremos metadata, outcomes/leakage y candidate features.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "dataset" / "final" / "creative_feature_base_tabular_visual_cv.parquet").exists():
    ROOT = ROOT.parent

DATASET_DIR = ROOT / "dataset"
OUTPUT_DIR = DATASET_DIR / "output"
FINAL_DIR = DATASET_DIR / "final"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)
INPUT_PATH = OUTPUT_DIR / "creative_feature_base_tabular_visual_cv.parquet"

df = pd.read_parquet(INPUT_PATH)
df.shape


## 1. Perfil general

Primero comprobamos grano, nulos, constantes y familias de columnas.

In [ ]:
if df["creative_id"].duplicated().any():
    raise ValueError("creative_id no es unico; no es una tabla de casos 1:1")

def family_for_col(col: str) -> str:
    if col.startswith("first_3d_"):
        return "early_3d"
    if col.startswith("first_7d_"):
        return "early_7d"
    if col.startswith("first_14d_"):
        return "early_14d"
    if col.startswith("lifecycle_"):
        return "full_lifecycle"
    if col.startswith("txtsel_"):
        return "text_engineered"
    if col.startswith("vis_"):
        return "fe_vision_handcrafted"
    if col.startswith("clip_pca_"):
        return "clip_pca"
    if col.startswith("cnn_pca_"):
        return "cnn_pca"
    if col.startswith("cv_"):
        return "cv_spatial"
    if col.startswith("prompt_"):
        return "cv_prompt_flat"
    if col.startswith("image_quality_"):
        return "image_quality"
    return "base_tabular"

profile = pd.DataFrame(
    {
        "rows": [len(df)],
        "columns": [df.shape[1]],
        "unique_creative_ids": [df["creative_id"].nunique()],
        "duplicate_creative_ids": [int(df["creative_id"].duplicated().sum())],
        "missing_cells": [int(df.isna().sum().sum())],
        "memory_mb": [df.memory_usage(deep=True).sum() / 1024**2],
    }
)

family_counts = pd.Series([family_for_col(c) for c in df.columns]).value_counts().rename_axis("family").reset_index(name="columns")
display(profile)
display(family_counts)


## 2. Roles: metadata, targets, leakage y features candidatas

Los targets/outcomes pueden quedarse en la tabla final de casos para explicar resultados historicos, pero no deben entrar como features para similitud predictiva o modelos.

In [ ]:
ID_COLS = [c for c in ["creative_id", "campaign_id", "advertiser_id"] if c in df.columns]
PATH_COLS = [c for c in ["asset_file"] if c in df.columns]
DISPLAY_TEXT_COLS = [c for c in ["headline", "subhead", "cta_text"] if c in df.columns]
DISPLAY_CONTEXT_COLS = [
    c for c in [
        "advertiser_name", "app_name", "vertical", "format", "language", "theme", "hook_type",
        "dominant_color", "emotional_tone", "objective", "primary_theme", "target_age_segment",
        "target_os", "kpi_goal", "hq_region", "creative_launch_date", "start_date", "end_date",
    ] if c in df.columns
]

TARGET_COLS = [
    c for c in [
        "creative_status", "fatigue_day", "has_fatigue", "perf_score",
        "overall_ctr", "overall_cvr", "overall_ipm", "overall_roas",
        "ctr_decay_pct", "cvr_decay_pct", "peak_rolling_ctr_5",
    ] if c in df.columns
]
FULL_PERIOD_COLS = [c for c in df.columns if c.startswith("lifecycle_")]
LEAKAGE_COLS = TARGET_COLS + FULL_PERIOD_COLS

EARLY_3D_COLS = [c for c in df.columns if c.startswith("first_3d_")]
EARLY_7D_COLS = [c for c in df.columns if c.startswith("first_7d_")]
EARLY_14D_COLS = [c for c in df.columns if c.startswith("first_14d_")]
EMBEDDING_PCA_COLS = [c for c in df.columns if c.startswith("clip_pca_") or c.startswith("cnn_pca_")]

METADATA_KEEP_COLS = list(dict.fromkeys(ID_COLS + PATH_COLS + DISPLAY_CONTEXT_COLS + DISPLAY_TEXT_COLS))

role_counts = pd.DataFrame(
    [
        {"role": "metadata_keep", "columns": len(METADATA_KEEP_COLS)},
        {"role": "target_outcome", "columns": len(TARGET_COLS)},
        {"role": "full_period_leakage", "columns": len(FULL_PERIOD_COLS)},
        {"role": "early_3d", "columns": len(EARLY_3D_COLS)},
        {"role": "early_7d", "columns": len(EARLY_7D_COLS)},
        {"role": "early_14d", "columns": len(EARLY_14D_COLS)},
        {"role": "embedding_pca", "columns": len(EMBEDDING_PCA_COLS)},
    ]
)
role_counts


## 3. Auditoria de columnas

Calculamos missing, cardinalidad, constantes, alta cardinalidad, y asignamos una recomendacion inicial.

In [ ]:
n = len(df)
constant_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
all_missing_cols = [c for c in df.columns if df[c].isna().all()]
high_missing_cols = [c for c in df.columns if df[c].isna().mean() > 0.50]

def base_role(col: str) -> str:
    if col in TARGET_COLS:
        return "target_outcome"
    if col in FULL_PERIOD_COLS:
        return "full_period_leakage"
    if col in ID_COLS:
        return "entity_key"
    if col in PATH_COLS:
        return "asset_reference"
    if col in DISPLAY_TEXT_COLS:
        return "display_text"
    if col in DISPLAY_CONTEXT_COLS:
        return "display_context_or_categorical"
    if col in EMBEDDING_PCA_COLS:
        return "embedding_pca_feature"
    if col in EARLY_3D_COLS:
        return "early_3d_feature"
    if col in EARLY_7D_COLS:
        return "early_7d_feature"
    if col in EARLY_14D_COLS:
        return "early_14d_feature"
    return "candidate_feature"

audit_rows = []
for col in df.columns:
    nunique = df[col].nunique(dropna=False)
    missing = int(df[col].isna().sum())
    audit_rows.append(
        {
            "column": col,
            "family": family_for_col(col),
            "role": base_role(col),
            "dtype": str(df[col].dtype),
            "missing_count": missing,
            "missing_pct": missing / n,
            "nunique": int(nunique),
            "unique_ratio": nunique / n,
            "is_constant": col in constant_cols,
            "is_all_missing": col in all_missing_cols,
            "is_high_missing": col in high_missing_cols,
        }
    )

column_audit = pd.DataFrame(audit_rows)

def initial_recommendation(row: pd.Series) -> str:
    col = row["column"]
    if row["is_all_missing"]:
        return "drop_all_missing"
    if row["is_constant"] and col not in ID_COLS:
        return "drop_constant"
    if col in TARGET_COLS:
        return "keep_as_target_not_feature"
    if col in FULL_PERIOD_COLS:
        return "keep_as_outcome_not_feature"
    if col in ID_COLS + PATH_COLS:
        return "keep_as_metadata_not_feature"
    if row["is_high_missing"]:
        return "review_high_missing"
    if row["dtype"] in ["object", "str", "string"] and row["unique_ratio"] > 0.50:
        return "keep_for_display_not_feature"
    return "candidate_feature"

column_audit["initial_recommendation"] = column_audit.apply(initial_recommendation, axis=1)
display(column_audit["initial_recommendation"].value_counts().rename_axis("recommendation").reset_index(name="columns"))
display(column_audit.sort_values(["initial_recommendation", "family", "column"]).head(40))


## 4. Redundancia exacta y correlacion alta

Buscamos columnas exactamente duplicadas y pares numericos con correlacion Spearman muy alta. Esto sirve para reducir dimensionalidad sin perder mucha informacion interpretable.

In [ ]:
hash_groups: dict[tuple[int, ...], list[str]] = {}
for col in df.columns:
    hashed = tuple(pd.util.hash_pandas_object(df[col], index=False).to_numpy().tolist())
    hash_groups.setdefault(hashed, []).append(col)

exact_duplicate_groups = [cols for cols in hash_groups.values() if len(cols) > 1]
exact_duplicate_rows = []
exact_duplicate_drop_cols = []
for group_id, cols in enumerate(exact_duplicate_groups, start=1):
    keep = cols[0]
    for drop_col in cols[1:]:
        exact_duplicate_drop_cols.append(drop_col)
        exact_duplicate_rows.append({"group_id": group_id, "keep_column": keep, "drop_column": drop_col, "group_columns": ", ".join(cols)})

exact_duplicate_report = pd.DataFrame(exact_duplicate_rows)

numeric_cols = df.select_dtypes(include=["number", "bool"]).columns.tolist()
protected = set(ID_COLS + LEAKAGE_COLS + constant_cols + all_missing_cols)
numeric_candidate_cols = [c for c in numeric_cols if c not in protected]

corr = df[numeric_candidate_cols].corr(method="spearman").abs()
upper_mask = np.triu(np.ones(corr.shape), k=1).astype(bool)
high_corr_pairs = (
    corr.where(upper_mask)
    .stack()
    .reset_index()
    .rename(columns={"level_0": "col_a", "level_1": "col_b", 0: "abs_spearman"})
    .query("abs_spearman >= 0.985")
    .sort_values("abs_spearman", ascending=False)
    .reset_index(drop=True)
)

correlated_drop_cols = []
already_dropped = set()
for row in high_corr_pairs.itertuples(index=False):
    a, b = row.col_a, row.col_b
    if a in already_dropped or b in already_dropped:
        continue
    # Preferimos conservar columnas base/interpretables y soltar duplicados derivados o prompt/cv redundantes cuando aparezcan como segundo termino.
    drop_col = b
    already_dropped.add(drop_col)
    correlated_drop_cols.append(drop_col)

display(pd.DataFrame({"exact_duplicate_groups": [len(exact_duplicate_groups)], "exact_duplicate_drop_cols": [len(exact_duplicate_drop_cols)], "high_corr_pairs": [len(high_corr_pairs)], "greedy_corr_drop_cols": [len(correlated_drop_cols)]}))
display(exact_duplicate_report.head(30))
display(high_corr_pairs.head(30))


## 5. Dataset candidato para CBR

Construimos un dataset de casos que conserva metadata y outcomes para explicar, pero define listas separadas de features que si pueden usarse para similitud. No se escalan ni se aplica PCA aqui: eso debe hacerse despues del split o dentro del pipeline de CBR.

In [ ]:
DROP_ALWAYS = set(all_missing_cols + constant_cols + exact_duplicate_drop_cols)
DROP_FOR_COMPACT_FEATURES = DROP_ALWAYS.union(correlated_drop_cols)

outcome_keep_cols = list(dict.fromkeys(TARGET_COLS + FULL_PERIOD_COLS))

candidate_feature_cols = []
for col in df.columns:
    if col in DROP_FOR_COMPACT_FEATURES:
        continue
    if col in METADATA_KEEP_COLS or col in outcome_keep_cols:
        continue
    if col in ID_COLS + PATH_COLS:
        continue
    candidate_feature_cols.append(col)

numeric_feature_cols = [c for c in candidate_feature_cols if c in numeric_cols]
categorical_feature_cols = [c for c in candidate_feature_cols if c not in numeric_cols]
prelaunch_feature_cols = [c for c in candidate_feature_cols if not c.startswith(("first_3d_", "first_7d_", "first_14d_"))]
early_3d_feature_cols = [c for c in candidate_feature_cols if not c.startswith(("first_7d_", "first_14d_"))]
early_7d_feature_cols = [c for c in candidate_feature_cols if not c.startswith("first_14d_")]
early_14d_feature_cols = candidate_feature_cols.copy()

final_case_cols = list(dict.fromkeys(METADATA_KEEP_COLS + outcome_keep_cols + candidate_feature_cols))
cbr_cases_final = df[final_case_cols].copy()

feature_sets = {
    "metadata_keep_cols": METADATA_KEEP_COLS,
    "target_cols": TARGET_COLS,
    "full_period_outcome_cols": FULL_PERIOD_COLS,
    "drop_always_cols": sorted(DROP_ALWAYS),
    "drop_correlated_cols": correlated_drop_cols,
    "candidate_feature_cols": candidate_feature_cols,
    "numeric_feature_cols": numeric_feature_cols,
    "categorical_feature_cols": categorical_feature_cols,
    "embedding_pca_cols": [c for c in candidate_feature_cols if c.startswith(("clip_pca_", "cnn_pca_"))],
    "prelaunch_feature_cols": prelaunch_feature_cols,
    "early_3d_feature_cols": early_3d_feature_cols,
    "early_7d_feature_cols": early_7d_feature_cols,
    "early_14d_feature_cols": early_14d_feature_cols,
}

pd.DataFrame(
    [
        {"set": name, "columns": len(cols)}
        for name, cols in feature_sets.items()
    ]
)


## 6. Guardado de artefactos

El dataset final candidato conserva outcomes para explicabilidad historica. Las columnas de features estan en `cbr_feature_sets.json`; usalas para no mezclar targets/leakage por accidente.

In [ ]:
column_audit["exact_duplicate_drop"] = column_audit["column"].isin(exact_duplicate_drop_cols)
column_audit["high_corr_drop_candidate"] = column_audit["column"].isin(correlated_drop_cols)
column_audit["in_cbr_final"] = column_audit["column"].isin(final_case_cols)
column_audit["is_candidate_similarity_feature"] = column_audit["column"].isin(candidate_feature_cols)

reduction_summary = pd.DataFrame(
    [
        {"metric": "input_columns", "value": df.shape[1]},
        {"metric": "drop_always_columns", "value": len(DROP_ALWAYS)},
        {"metric": "drop_correlated_candidates", "value": len(correlated_drop_cols)},
        {"metric": "final_case_columns", "value": cbr_cases_final.shape[1]},
        {"metric": "candidate_similarity_features", "value": len(candidate_feature_cols)},
        {"metric": "numeric_similarity_features", "value": len(numeric_feature_cols)},
        {"metric": "categorical_similarity_features", "value": len(categorical_feature_cols)},
        {"metric": "prelaunch_features", "value": len(prelaunch_feature_cols)},
        {"metric": "early_3d_features", "value": len(early_3d_feature_cols)},
        {"metric": "early_7d_features", "value": len(early_7d_feature_cols)},
        {"metric": "early_14d_features", "value": len(early_14d_feature_cols)},
    ]
)

column_audit.to_csv(OUTPUT_DIR / "advanced_eda_column_audit.csv", index=False)
exact_duplicate_report.to_csv(OUTPUT_DIR / "advanced_eda_exact_duplicate_columns.csv", index=False)
high_corr_pairs.to_csv(OUTPUT_DIR / "advanced_eda_high_corr_pairs.csv", index=False)
reduction_summary.to_csv(OUTPUT_DIR / "advanced_eda_reduction_summary.csv", index=False)
cbr_cases_final.to_parquet(FINAL_DIR / "creative_cbr_cases_final.parquet", index=False)
cbr_cases_final.to_csv(FINAL_DIR / "creative_cbr_cases_final.csv", index=False)

with (FINAL_DIR / "cbr_feature_sets.json").open("w", encoding="utf-8") as handle:
    json.dump(feature_sets, handle, indent=2, ensure_ascii=True)

with (OUTPUT_DIR / "advanced_eda_summary.json").open("w", encoding="utf-8") as handle:
    json.dump(
        {
            "input_path": str(INPUT_PATH),
            "input_shape": list(df.shape),
            "final_cbr_cases_path": str(FINAL_DIR / "creative_cbr_cases_final.parquet"),
            "final_cbr_cases_shape": list(cbr_cases_final.shape),
            "recommendation": "Use creative_cbr_cases_final as the case table; use cbr_feature_sets.json to choose feature columns per prelaunch/early window. Fit scalers/PCA after split, not on the full dataset.",
        },
        handle,
        indent=2,
        ensure_ascii=True,
    )

display(reduction_summary)
display(pd.DataFrame({"artifact": [
    "advanced_eda_column_audit.csv",
    "advanced_eda_exact_duplicate_columns.csv",
    "advanced_eda_high_corr_pairs.csv",
    "advanced_eda_reduction_summary.csv",
    "creative_cbr_cases_final.parquet",
    "creative_cbr_cases_final.csv",
    "cbr_feature_sets.json",
    "advanced_eda_summary.json",
]}))


## 7. Lectura practica

- Para CBR, usar `creative_cbr_cases_final.parquet` como tabla de casos.
- Para similitud prelaunch, usar `prelaunch_feature_cols` desde `cbr_feature_sets.json`.
- Para similitud tras 7 dias, usar `early_7d_feature_cols`.
- No usar `target_cols` ni `full_period_outcome_cols` para calcular similitud predictiva.
- Si se aplica scaler/PCA, ajustarlo despues del split. Para una demo sin evaluacion, se puede ajustar sobre toda la memoria historica, pero hay que explicarlo como indexacion de memoria, no benchmark predictivo.